# Giải đấu chính thức – Chế độ B trên Google Colab (issue #15)

Chế độ B dùng cấu hình cố định (Alpha-Beta độ sâu 3, MCTS 1.600 mô phỏng, PUCT 400 mô phỏng, cấp 5–6 argmax policy), nên **kết quả không phụ thuộc máy**: chạy trên Colab hay laptop đều ra cùng các ván, chỉ khác tốc độ. Chế độ A (theo giây) **không** chạy ở đây mà chạy trên một laptop.

**Chuẩn bị (1 lần):** tải `Nhom31_GiaiDau_CheDoB.zip` lên Google Drive, đặt đúng đường dẫn `MyDrive/caro/Nhom31_GiaiDau_CheDoB.zip`.

**Cách chạy:**
1. **Runtime → Change runtime type → CPU** (không cần GPU: giải đấu gần như chỉ dùng CPU, xem mục 3.5 báo cáo).
2. Sửa `PART` / `TOTAL_PARTS` ở ô cấu hình nếu được giao một phần (mặc định 1/1 = chạy hết).
3. **Runtime → Run all.**
4. **Bị ngắt** (hết 12 giờ, mất mạng, để yên quá lâu): mở lại notebook và **Run all** lại. Log nằm trên Drive nên chương trình tự bỏ qua các ván đã xong và chạy tiếp.

Log ghi vào `MyDrive/caro/giai_dau/mode_b/part<PART>of<TOTAL_PARTS>/`. Xong thì báo Hiếu, gửi thư mục đó.

## 1. Cấu hình phần được giao

In [ ]:
PART = 1           # phần được giao: 1..TOTAL_PARTS
TOTAL_PARTS = 1    # tổng số phần (số tài khoản Colab chạy song song); mọi người phải dùng CÙNG con số này
ZIP_PATH = '/content/drive/MyDrive/caro/Nhom31_GiaiDau_CheDoB.zip'
assert 1 <= PART <= TOTAL_PARTS
LOG_DIR = f'/content/drive/MyDrive/caro/giai_dau/mode_b/part{PART}of{TOTAL_PARTS}'
print('Phần', PART, '/', TOTAL_PARTS, '-> log:', LOG_DIR)

## 2. Mount Drive, giải nén code + checkpoint, cài thư viện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
assert os.path.exists(ZIP_PATH), f'Không thấy {ZIP_PATH} – tải file ZIP lên Drive đúng chỗ này'
!rm -rf /content/goi && mkdir -p /content/goi && unzip -q -o "$ZIP_PATH" -d /content/goi
%cd /content/goi/Nhom31_GiaiDau_CheDoB/do-an-caro
!cat COMMIT.txt
!sha256sum -c <(tail -n +2 COMMIT.txt)
# Colab có sẵn torch/numpy/pandas; chỉ cài phần còn thiếu, không cài lại torch
!pip install -q "numba>=0.58" "pyyaml>=6.0"
os.makedirs(LOG_DIR, exist_ok=True)
print('CPU:', os.cpu_count(), 'vCPU')

## 3. Tạo config cho phần được giao

28 cặp được chia theo **khối lượng ước tính** (các cặp có MCTS 1.600 mô phỏng chạy 3 seed nên nặng nhất) để các phần chạy lâu gần bằng nhau. Cách chia là tất định: ai chạy với cùng `TOTAL_PARTS` cũng ra đúng cùng danh sách, không trùng và không sót cặp.

In [ ]:
import itertools, yaml
cfg = yaml.safe_load(open('configs/arena/mode_b.yaml'))
labels = list(cfg['agents'])
# thời gian tương đối mỗi nước (đo trên CPU, chế độ B)
cost = {'random': 0.0, 'heuristic': 0.03, 'alphabeta': 0.2, 'mcts': 1.0,
        'sl': 0.003, 'rl': 0.003, 'puct': 0.5, 'alphazero': 0.6}
stoch = {'random', 'mcts'}
pairs = list(itertools.combinations(labels, 2))
def weight(p):
    games = 200 * (len(cfg['seeds']) if (p[0] in stoch or p[1] in stoch) else 1)
    return games * (cost[p[0]] + cost[p[1]] + 0.01)
parts = [[] for _ in range(TOTAL_PARTS)]; load = [0.0] * TOTAL_PARTS
for p in sorted(pairs, key=lambda p: (-weight(p), p)):
    i = min(range(TOTAL_PARTS), key=lambda k: (load[k], k)); parts[i].append(list(p)); load[i] += weight(p)
for i, ps in enumerate(parts, 1):
    print(f'Phần {i}: {len(ps)} cặp, tải tương đối {load[i-1]:.0f}' + ('   <- BẠN' if i == PART else ''))
cfg['pairs'] = parts[PART - 1]
cfg['workers'] = os.cpu_count()
yaml.safe_dump(cfg, open('/content/mode_b_part.yaml', 'w'), allow_unicode=True, sort_keys=False)
print('\nCác cặp của bạn:'); [print('  ', a, 'vs', b) for a, b in cfg['pairs']]

## 4. Chạy giải đấu (bị ngắt thì Run all lại, tự chạy tiếp)

In [ ]:
!python -m arena.tournament /content/mode_b_part.yaml --log-dir "$LOG_DIR"

## 5. Kiểm tra tiến độ / kết quả

In [ ]:
import json, collections
log = os.path.join(LOG_DIR, cfg['name'] + '.jsonl')
games = [json.loads(l) for l in open(log)] if os.path.exists(log) else []
cnt = collections.Counter(g['game_id'].rsplit('__s', 1)[0] for g in games)
print(f'Đã xong {len(games)} ván')
for a, b in cfg['pairs']:
    print(f'  {a} vs {b}: {cnt.get(f"{a}__{b}", 0)} ván')
print('Lỗi tác tử:', sum(1 for g in games if g.get('error')))